# ARGUS Colab: 58-Day Resumable Pipeline

This notebook runs days 1-58 with checkpoint-safe resume behavior using Google Drive persistence.

In [ ]:
# --- Parameters ---
AUTH_PATH = None  # Auto-resolved from kagglehub unless manually set to an existing file
OUTPUT_ROOT = "/content/drive/MyDrive/argus_outputs"
START_DAY = 1
END_DAY = 58
DAY_BATCH_SIZE = 2  # Colab is less stable for long runs; smaller batches are safer
BUCKET_COUNT = 256  # Higher bucket_count lowers peak RAM in build_sessions.py
TOKENIZE_PARQUET_BATCH_SIZE = 1000  # Lower this if tokenization still gets close to RAM limits
TOKENIZED_CHUNK_SIZE = 10000  # Larger chunks reduce tiny-file pressure on Google Drive
TOKENIZE_MAX_LEN = 128  # Honest Colab/Drive default. Use 512 only if you accept a very long, large Drive write.
TOKEN_ID_DTYPE = "int16"  # Use int32 if tokenization says the vocab is too large for int16
ATTENTION_MASK_DTYPE = "bool"
RESUME = True

# Dataset setup (Colab)
KAGGLE_DATASET_REF = "poornimakodithuwakku/lanl-dataset"
KAGGLEHUB_AUTH_FILENAME = "auth.txt"

# Repo setup options
USE_GIT_CLONE = True
REFRESH_GIT_CLONE = True  # Re-clone by default so Colab does not reuse a stale build_sessions.py
REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = "/content/ARGUS"
ZIP_PATH = "/content/ARGUS.zip"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and USE_GIT_CLONE and REFRESH_GIT_CLONE:
    print(f"Refreshing git clone at: {repo_dir}")
    shutil.rmtree(repo_dir)

if repo_dir.exists():
    print(f"Using existing repo at: {repo_dir}")
elif USE_GIT_CLONE:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    zip_path = Path(ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP_PATH not found: {zip_path}")
    extract_root = Path("/content/argus_zip_extract")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", str(zip_path), "-d", str(extract_root)], check=True)
    candidates = sorted({p.parent for p in extract_root.rglob("build_sessions.py") if p.parent.name == "scripts"})
    if not candidates:
        raise FileNotFoundError("Could not find scripts/build_sessions.py in uploaded zip.")
    project_root = candidates[0].parent
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    shutil.move(str(project_root), str(repo_dir))

if not (repo_dir / "scripts" / "build_sessions.py").exists():
    nested = repo_dir / "argus-log-intelligence-platform"
    if (nested / "scripts" / "build_sessions.py").exists():
        repo_dir = nested
        REPO_DIR = str(repo_dir)

BUILD_SESSIONS_SCRIPT = repo_dir / "scripts" / "build_sessions.py"
BUILD_TOKENIZE_SCRIPT = repo_dir / "scripts" / "build_vocab_and_tokenize.py"

def ensure_build_sessions_supports_bucket_count() -> Path:
    if not BUILD_SESSIONS_SCRIPT.exists():
        raise FileNotFoundError(f"build_sessions.py not found at: {BUILD_SESSIONS_SCRIPT}")

    probe = subprocess.run(
        [sys.executable, str(BUILD_SESSIONS_SCRIPT), "--help"],
        cwd=str(repo_dir),
        capture_output=True,
        text=True,
        check=False,
    )
    help_text = (probe.stdout or "") + (probe.stderr or "")
    if probe.returncode != 0:
        raise RuntimeError(
            "Failed to inspect build_sessions.py CLI. "
            f"script={BUILD_SESSIONS_SCRIPT} returncode={probe.returncode}\n{help_text.strip()}"
        )
    if "--bucket-count" not in help_text:
        raise RuntimeError(
            "Resolved build_sessions.py does not support --bucket-count. "
            f"The Colab repo copy is stale: {BUILD_SESSIONS_SCRIPT}. "
            "Refresh /content/ARGUS or rerun the repo setup cell with REFRESH_GIT_CLONE=True."
        )

    print(f"Validated build_sessions.py CLI: {BUILD_SESSIONS_SCRIPT}")
    return BUILD_SESSIONS_SCRIPT

def ensure_tokenizer_supports_drive_safe_flags() -> Path:
    if not BUILD_TOKENIZE_SCRIPT.exists():
        raise FileNotFoundError(f"build_vocab_and_tokenize.py not found at: {BUILD_TOKENIZE_SCRIPT}")

    probe = subprocess.run(
        [sys.executable, str(BUILD_TOKENIZE_SCRIPT), "--help"],
        cwd=str(repo_dir),
        capture_output=True,
        text=True,
        check=False,
    )
    help_text = (probe.stdout or "") + (probe.stderr or "")
    required_flags = ("--token-id-dtype", "--attention-mask-dtype", "--progress-interval-rows")
    missing = [flag for flag in required_flags if flag not in help_text]
    if probe.returncode != 0 or missing:
        raise RuntimeError(
            "Resolved build_vocab_and_tokenize.py is stale and does not support the Drive-safe tokenization flags. "
            f"Missing flags: {missing}. Script: {BUILD_TOKENIZE_SCRIPT}. "
            "Push/pull the latest repo changes, or set USE_GIT_CLONE=False and upload a fresh zip."
        )

    print(f"Validated build_vocab_and_tokenize.py CLI: {BUILD_TOKENIZE_SCRIPT}")
    return BUILD_TOKENIZE_SCRIPT

print(f"Repo ready: {repo_dir}")


In [ ]:
import subprocess
import sys

# Install only the pipeline dependencies needed for sessionization/tokenization in this notebook.
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pandas",
    "pyarrow",
    "tqdm",
    "drain3",
    "jsonpickle",
    "torch",
], check=True)
print("Pipeline dependencies installed.")


In [ ]:
import subprocess
import sys
from pathlib import Path

if AUTH_PATH and Path(AUTH_PATH).exists():
    print(f"Using user-provided AUTH_PATH: {AUTH_PATH}")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub
    dataset_root = Path(kagglehub.dataset_download(KAGGLE_DATASET_REF))
    matches = sorted(dataset_root.rglob(KAGGLEHUB_AUTH_FILENAME))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {KAGGLEHUB_AUTH_FILENAME} under downloaded dataset path: {dataset_root}"
        )
    AUTH_PATH = str(matches[0])
    print(f"kagglehub dataset root: {dataset_root}")
    print(f"Resolved AUTH_PATH: {AUTH_PATH}")
    if len(matches) > 1:
        print(f"Found {len(matches)} auth.txt matches. Using first: {AUTH_PATH}")


In [ ]:
# Optional smoke test before full-data run.
# Set RUN_SMOKE_TEST = True to validate environment quickly.
RUN_SMOKE_TEST = False

import subprocess
import sys
from pathlib import Path

if RUN_SMOKE_TEST:
    build_sessions_script = ensure_build_sessions_supports_bucket_count()
    sample_input = Path(REPO_DIR) / "data" / "raw" / "auth_sample_200k.txt"
    smoke_root = Path(OUTPUT_ROOT) / "smoke_test"
    (smoke_root / "data" / "sessions").mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(build_sessions_script),
        "--input",
        str(sample_input),
        "--start-day",
        "1",
        "--end-day",
        "1",
        "--output-dir",
        str(smoke_root / "data" / "sessions"),
        "--parser-state",
        str(smoke_root / "data" / "drain3_state.bin"),
        "--bucket-count",
        str(BUCKET_COUNT),
    ]
    print(f"Smoke test script: {build_sessions_script}")
    subprocess.run(cmd, check=True)
    print("Smoke test complete.")
else:
    print("Skipping smoke test.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pyarrow.parquet as pq

output_root = Path(OUTPUT_ROOT)
data_root = output_root / "data"
sessions_dir = data_root / "sessions"
tokenized_dir = data_root / "tokenized"

sessions_dir.mkdir(parents=True, exist_ok=True)
tokenized_dir.mkdir(parents=True, exist_ok=True)
os.chdir(output_root)

def shard_ready(day: int) -> bool:
    shard = sessions_dir / f"day_{day:02d}.parquet"
    if not shard.exists() or shard.stat().st_size <= 0:
        return False
    try:
        pq.ParquetFile(shard)
    except Exception as exc:
        print(f"[WARN] day {day:02d} shard is unreadable and will be rebuilt: {exc}")
        shard.unlink(missing_ok=True)
        return False
    return True

build_sessions_script = ensure_build_sessions_supports_bucket_count()
print(f"Using build_sessions.py: {build_sessions_script}")

ready_days = [day for day in range(START_DAY, END_DAY + 1) if shard_ready(day)]
not_ready_days = [day for day in range(START_DAY, END_DAY + 1) if day not in set(ready_days)]
print(f"Ready session shards: {len(ready_days)}/{END_DAY - START_DAY + 1}")
if not_ready_days:
    print(f"Days to build/rebuild: {not_ready_days[:20]}{' ...' if len(not_ready_days) > 20 else ''}")

batch_size = max(1, int(DAY_BATCH_SIZE))
day = START_DAY

while day <= END_DAY:
    shard = sessions_dir / f"day_{day:02d}.parquet"
    if RESUME and shard_ready(day):
        print(f"[SKIP] day {day:02d} already exists ({shard.stat().st_size} bytes)")
        day += 1
        continue

    batch_start = day
    batch_end = min(day + batch_size - 1, END_DAY)

    if RESUME:
        cursor = batch_start
        while cursor <= batch_end and not shard_ready(cursor):
            cursor += 1
        batch_end = cursor - 1

    if batch_end < batch_start:
        day += 1
        continue

    print(f"[RUN ] building days {batch_start:02d}-{batch_end:02d} (bucket_count={BUCKET_COUNT})")
    cmd = [
        sys.executable,
        str(build_sessions_script),
        "--input",
        AUTH_PATH,
        "--start-day",
        str(batch_start),
        "--end-day",
        str(batch_end),
        "--output-dir",
        "data/sessions",
        "--parser-state",
        "data/drain3_state.bin",
        "--bucket-count",
        str(BUCKET_COUNT),
    ]
    subprocess.run(cmd, check=True)
    day = batch_end + 1

print("Session shard pass complete.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(OUTPUT_ROOT)
tokenizer_script = ensure_tokenizer_supports_drive_safe_flags()
sessions_dir = Path(OUTPUT_ROOT) / "data" / "sessions"
session_shards = sorted(p for p in sessions_dir.glob("day_*.parquet") if p.is_file() and p.stat().st_size > 0)
print(f"Tokenizing from {len(session_shards)} existing session parquet shard(s): {sessions_dir}")
if len(session_shards) < END_DAY - START_DAY + 1:
    present_days = {
        int(p.stem.split("_")[-1])
        for p in session_shards
        if p.stem.split("_")[-1].isdigit()
    }
    missing_days = [day for day in range(START_DAY, END_DAY + 1) if day not in present_days]
    raise FileNotFoundError(
        f"Expected {END_DAY - START_DAY + 1} session shards before tokenization, "
        f"but found {len(session_shards)}. Missing days: {missing_days[:20]}"
        f"{' ...' if len(missing_days) > 20 else ''}. Run the session-building cell first."
    )

split_jobs = [
    ("train", "--vocab-out", "data/vocab.json", "data/tokenized/sessions_train.pt"),
    ("val", "--vocab-in", "data/vocab.json", "data/tokenized/sessions_val.pt"),
    ("test", "--vocab-in", "data/vocab.json", "data/tokenized/sessions_test.pt"),
]

for split, vocab_arg, vocab_path, token_out in split_jobs:
    print(f"[RUN ] tokenization split={split}")
    cmd = [
        sys.executable,
        str(tokenizer_script),
        "--sessions-glob",
        "data/sessions/day_*.parquet",
        "--split",
        split,
        vocab_arg,
        vocab_path,
        "--tokenized-out",
        token_out,
        "--parquet-batch-size",
        str(TOKENIZE_PARQUET_BATCH_SIZE),
        "--tokenized-chunk-size",
        str(TOKENIZED_CHUNK_SIZE),
        "--max-len",
        str(TOKENIZE_MAX_LEN),
        "--token-id-dtype",
        TOKEN_ID_DTYPE,
        "--attention-mask-dtype",
        ATTENTION_MASK_DTYPE,
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

print("Tokenization pass complete.")

In [ ]:
from pathlib import Path

sessions_dir = Path(OUTPUT_ROOT) / "data" / "sessions"
tokenized_dir = Path(OUTPUT_ROOT) / "data" / "tokenized"

session_files = sorted(sessions_dir.glob("day_*.parquet"))
print(f"Session shards present: {len(session_files)}")
if session_files:
    print(f"First shard: {session_files[0].name}")
    print(f"Last shard:  {session_files[-1].name}")

for path in [
    Path(OUTPUT_ROOT) / "data" / "vocab.json",
    tokenized_dir / "sessions_train.pt",
    tokenized_dir / "sessions_val.pt",
    tokenized_dir / "sessions_test.pt",
]:
    print(f"{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

for chunk_dir in sorted(tokenized_dir.glob("sessions_*_chunks")):
    chunk_count = len(list(chunk_dir.glob("chunk_*.pt")))
    print(f"{chunk_dir}: chunks={chunk_count}")